In [0]:
"""
Data Quality - Generate Column Summary Task
Calculates scores, statuses, and faulty counts per column
"""
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
import json

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("columns", "[]")
dbutils.widgets.text("enabled_dimensions", "[]")
dbutils.widgets.text("threshold_ok", "100.0")
dbutils.widgets.text("threshold_warning", "98.0")
dbutils.widgets.text("thresholds_json", "{}")

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
COLUMNS_TO_CHECK = json.loads(dbutils.widgets.get("columns"))
ENABLED_DIMENSIONS = json.loads(dbutils.widgets.get("enabled_dimensions"))
THRESHOLD_OK = float(dbutils.widgets.get("threshold_ok"))
THRESHOLD_WARNING = float(dbutils.widgets.get("threshold_warning"))
THRESHOLDS_CONFIG = json.loads(dbutils.widgets.get("thresholds_json") or "{}")

# Sanitize names for table naming
import re
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Generate Column Summary ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Results Schema: {RESULTS_SCHEMA}")
print(f"Columns to check: {COLUMNS_TO_CHECK}")
print(f"Enabled dimensions: {ENABLED_DIMENSIONS}")
print(f"Thresholds: OK ≥ {THRESHOLD_OK}%, Warning ≥ {THRESHOLD_WARNING}%")

# Read aggregated checks
agg_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_checks_aggregated"
print(f"\nReading from: {agg_table}")
df_agg = spark.table(agg_table)

total_rows = df_agg.count()
print(f"Total rows: {total_rows}")
print(f"Available columns in aggregated table: {df_agg.columns}")

# Define output schema
summary_schema = StructType([
    StructField("column_name", StringType(), False),
    StructField("check_type", StringType(), False),
    StructField("total_rows", LongType(), False),
    StructField("valid_count", LongType(), False),
    StructField("invalid_count", LongType(), False),
    StructField("null_count", LongType(), False),
    StructField("score", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("faulty_count", LongType(), True),
    StructField("threshold_ok", DoubleType(), False),
    StructField("threshold_warning", DoubleType(), False),
    StructField("execution_timestamp", StringType(), False)
])

# Collect summary data
summary_data = []
execution_timestamp = spark.sql("SELECT current_timestamp() as ts").collect()[0]['ts']

def calculate_score_and_status(valid, invalid, null_count):
    """Calculate score and determine status"""
    if valid + invalid == 0:
        return None, None
    score = (valid / (valid + invalid)) * 100
    if score >= THRESHOLD_OK:
        status = "OK"
    elif score >= THRESHOLD_WARNING:
        status = "Warning"
    else:
        status = "KO"
    return round(score, 2), status

def thresholds_for(dimension, column_name):
    configured = THRESHOLDS_CONFIG.get("by_dimension", {}).get(dimension, {}).get(column_name, {})
    return (
        float(configured.get("ok", THRESHOLD_OK)),
        float(configured.get("warning", THRESHOLD_WARNING))
    )

print("\nCalculating summaries:")

# Check if COLUMNS_TO_CHECK is empty
if not COLUMNS_TO_CHECK:
    print("WARNING: No columns specified in COLUMNS_TO_CHECK parameter!")
    print("Attempting to infer columns from aggregated table...")
    # Extract column names from check columns
    check_cols = [col for col in df_agg.columns if col.endswith('_check')]
    print(f"Found {len(check_cols)} check columns: {check_cols}")
    
    # Extract unique base column names
    inferred_columns = set()
    deduped_dimensions = list(dict.fromkeys(ENABLED_DIMENSIONS + ['accuracy', 'overall']))
    for check_col in check_cols:
        # Remove dimension suffix (_uniqueness_check, _completeness_check, _accuracy_check, _overall_check, etc.)
        for dim in deduped_dimensions:
            if check_col.endswith(f"_{dim}_check"):
                base_col = check_col[:-len(f"_{dim}_check")]
                inferred_columns.add(base_col)
                break
    
    COLUMNS_TO_CHECK = sorted(list(inferred_columns))
    print(f"Inferred columns: {COLUMNS_TO_CHECK}")

# Process each column and dimension
for col_name in COLUMNS_TO_CHECK:
    # Include all enabled dimensions + accuracy + overall, deduplicated
    all_dimensions = list(dict.fromkeys(ENABLED_DIMENSIONS + ['accuracy', 'overall']))
    for dimension in all_dimensions:
        check_col = f"{col_name}_{dimension}_check"
        
        if check_col not in df_agg.columns:
            print(f"  ⊘ {check_col} not found in aggregated table")
            continue
        
        print(f"  Processing {check_col}...")
        
        # Calculate counts
        valid_count = df_agg.filter(F.col(check_col) == 1).count()
        invalid_count = df_agg.filter(F.col(check_col) == 0).count()
        null_count = df_agg.filter(F.col(check_col).isNull()).count()
        
        # Verify counts sum to total
        count_sum = valid_count + invalid_count + null_count
        if count_sum != total_rows:
            print(f"  ⚠ WARNING: {col_name} - {dimension}: Count mismatch! {count_sum} != {total_rows}")
            print(f"    valid={valid_count}, invalid={invalid_count}, null={null_count}")
        
        # Calculate score and status
        threshold_ok, threshold_warning = thresholds_for(dimension, col_name)
        score, status = calculate_score_and_status(valid_count, invalid_count, null_count)
        if score is not None:
            if score >= threshold_ok:
                status = "OK"
            elif score >= threshold_warning:
                status = "Warning"
            else:
                status = "KO"
        
        if score is not None:
            print(f"  ✓ {col_name} - {dimension}: {score}% ({status}) - Valid: {valid_count}, Invalid: {invalid_count}, Null: {null_count}")
            
            summary_data.append((
                col_name,
                dimension,
                total_rows,
                valid_count,
                invalid_count,
                null_count,
                score,
                status,
                invalid_count,  # faulty_count same as invalid_count
                threshold_ok,
                threshold_warning,
                str(execution_timestamp)
            ))
        else:
            print(f"  ⊘ {col_name} - {dimension}: No valid data (valid={valid_count}, invalid={invalid_count}, null={null_count})")

print(f"\nTotal summary rows created: {len(summary_data)}")

# Create summary DataFrame
df_summary = spark.createDataFrame(summary_data, summary_schema)

# Output table name
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_summary_columns"

print(f"\nWriting to: {output_table}")

# Write to Unity Catalog
df_summary.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✅ Successfully wrote {df_summary.count()} rows to {output_table}")

# Display summary
print("\nSummary (first 10 rows):")
df_summary.show(10, truncate=False)